1.SET UP

In [1]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Disable warnings
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)
import seaborn as sns

%matplotlib inline

2. DATA CLEANING

In [2]:
df = pd.read_csv(('zillow_10_thanhpho.csv'),sep=',',on_bad_lines='skip',  engine="python")
df.head()

,zpid,palsId,rawHomeStatusCd,marketingStatusSimplifiedCd,imgSrc,hasImage,detailUrl,statusType,statusText,price,...,builderName,communityName,isPropertyResultCDP,isCdpResult,isComingSoonCommunity,hdpData.homeInfo.group_type,hdpData.homeInfo.priceSuffix,hdpData.homeInfo.providerListingID,style,info2String
0,32374118.0,820002_2603210,ForSale,For Sale by Agent,https://photos.zillowstatic.com/fp/d493281aea2...,True,/homedetails/51-Outerbridge-Ave-Staten-Island-...,FOR_SALE,Multi-family home for sale,"$1,579,000",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,32374450.0,479004_501930,ForSale,For Sale by Agent,https://photos.zillowstatic.com/fp/06a143e7422...,True,/homedetails/127-Madsen-Ave-Staten-Island-NY-1...,FOR_SALE,Multi-family home for sale,"$1,299,000",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,32333788.0,820001_2603272,ForSale,For Sale by Agent,https://photos.zillowstatic.com/fp/47a42ffca54...,True,/homedetails/25-Peggy-Ln-Staten-Island-NY-1030...,FOR_SALE,Townhouse for sale,"$599,900",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,299993810.0,820002_2603046,ForSale,For Sale by Agent,https://photos.zillowstatic.com/fp/9739c199f4f...,True,/homedetails/29-Burgher-Ave-Staten-Island-NY-1...,FOR_SALE,Multi-family home for sale,"$1,388,000",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,32296588.0,820001_2603293,ForSale,For Sale by Agent,https://photos.zillowstatic.com/fp/a4b3062667d...,True,/homedetails/155-Decker-Ave-Staten-Island-NY-1...,FOR_SALE,House for sale,"$699,900",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Length of dataset
len(df)

46078

In [4]:
nan_value = float("NaN")
df.replace("0", nan_value, inplace=True)
# Quick stats of percentage of null values in columns
(df.isnull().sum() / df.isnull().count()).sort_values(ascending=False)

hdpData.homeInfo.priceSuffix                        0.999783
hdpData.homeInfo.group_type                         0.999783
isPropertyResultCDP                                 0.999783
style                                               0.998242
hdpData.homeInfo.listing_sub_type.is_foreclosure    0.998025
                                                      ...   
latLong.longitude                                   0.000000
statusType                                          0.000000
detailUrl                                           0.000000
imgSrc                                              0.000000
latLong.latitude                                    0.000000
Length: 113, dtype: float64

In [5]:
# Drop columns with over 50% of null values
thresh = len(df) * .5
df.dropna(thresh = thresh, axis = 1, inplace = True)

In [6]:
# Same for numeric columns with too many 0 and -1 values (also null values of numeric columns but turned to 0 and -1 by crawling function)
drop_cols = df.columns[(df.isin([0, -1]).mean() > 0.5)]
df.drop(columns=drop_cols, inplace=True)

In [7]:
# Drop duplicate columns(same values)
drop_cols = []
for i in range(len(df.columns)):
    for j in range(i+1, len(df.columns)):
        if df[df.columns[i]].equals(df[df.columns[j]]):
            drop_cols.append(df.columns[j])

df.drop(drop_cols, axis=1, inplace=True)

In [8]:
#Null percentage after dropping
(df.isnull().sum() / df.isnull().count()).sort_values(ascending=False)

hdpData.homeInfo.zestimate                   0.438365
hdpData.homeInfo.rentZestimate               0.325058
hdpData.homeInfo.lotAreaUnit                 0.231520
hdpData.homeInfo.lotAreaValue                0.231520
hdpData.homeInfo.taxAssessedValue            0.193281
hdpData.homeInfo.listing_sub_type.is_FSBA    0.190655
contentType                                  0.164460
flexFieldText                                0.164460
brokerName                                   0.137354
hdpData.homeInfo.timeOnZillow                0.128760
area                                         0.120752
baths                                        0.107470
beds                                         0.105387
hdpData.homeInfo.price                       0.097530
hdpData.homeInfo.latitude                    0.097053
hdpData.homeInfo.state                       0.097053
hdpData.homeInfo.longitude                   0.097053
hdpData.homeInfo.homeType                    0.097053
hdpData.homeInfo.zipcode    

In [9]:
df.columns


Index(['zpid', 'palsId', 'rawHomeStatusCd', 'marketingStatusSimplifiedCd',
       'imgSrc', 'hasImage', 'detailUrl', 'statusType', 'statusText', 'price',
       'priceLabel', 'address', 'addressState', 'beds', 'baths', 'area',
       'flexFieldText', 'contentType', 'sgapt', 'hasAdditionalAttributions',
       'brokerName', 'timeOnZillow', 'latLong.latitude', 'latLong.longitude',
       'hdpData.homeInfo.streetAddress', 'hdpData.homeInfo.zipcode',
       'hdpData.homeInfo.city', 'hdpData.homeInfo.state',
       'hdpData.homeInfo.latitude', 'hdpData.homeInfo.longitude',
       'hdpData.homeInfo.price', 'hdpData.homeInfo.homeType',
       'hdpData.homeInfo.homeStatus', 'hdpData.homeInfo.zestimate',
       'hdpData.homeInfo.rentZestimate',
       'hdpData.homeInfo.listing_sub_type.is_FSBA',
       'hdpData.homeInfo.timeOnZillow', 'hdpData.homeInfo.isNonOwnerOccupied',
       'hdpData.homeInfo.currency', 'hdpData.homeInfo.country',
       'hdpData.homeInfo.taxAssessedValue', 'hdpData.homeIn

In [10]:
len(df.columns)

43

In [11]:
#Picking useable columns:
usefull_cols = [
    'hdpData.homeInfo.price', 
    'statusText', 
    'hdpData.homeInfo.homeType',
    'brokerName', 
    'hasAdditionalAttributions', 
    'hdpData.homeInfo.city',
    'hdpData.homeInfo.state', 
    'hdpData.homeInfo.latitude', 
    'hdpData.homeInfo.longitude',
    'baths', 
    'beds', 
    'area',
    'hdpData.homeInfo.lotAreaValue', 
    'hdpData.homeInfo.lotAreaUnit',
    'hdpData.homeInfo.taxAssessedValue', 
    'hdpData.homeInfo.zestimate', 
    'hdpData.homeInfo.rentZestimate',
    'hdpData.homeInfo.listing_sub_type.is_FSBA', 
    'hdpData.homeInfo.isNonOwnerOccupied', 
    'timeOnZillow'
]

dff = df[usefull_cols]

In [12]:
dff.rename(columns={
    'hdpData.homeInfo.price': 'price',
    'statusText': 'status',
    'hdpData.homeInfo.homeType': 'type',
    'brokerName': 'broker_name',
    'hasAdditionalAttributions': 'has_add_attributions',
    'hdpData.homeInfo.city': 'city',
    'hdpData.homeInfo.state': 'state',
    'hdpData.homeInfo.latitude': 'latitude',
    'hdpData.homeInfo.longitude': 'longitude',
    'baths': 'bathrooms',
    'beds': 'bedrooms',
    'area': 'living_area',
    'hdpData.homeInfo.lotAreaValue': 'lot_area',
    'hdpData.homeInfo.lotAreaUnit': 'lot_area_unit',
    'hdpData.homeInfo.taxAssessedValue': 'tax_assessed_value',
    'hdpData.homeInfo.zestimate': 'zestimate',
    'hdpData.homeInfo.rentZestimate': 'rent_zestimate',
    'hdpData.homeInfo.listing_sub_type.is_FSBA': 'is_fsba',
    'hdpData.homeInfo.isNonOwnerOccupied': 'is_non_owner_occupied',
    'timeOnZillow': 'days_on_zillow'
}, inplace=True)

dff.columns

Index(['price', 'status', 'type', 'broker_name', 'has_add_attributions',
       'city', 'state', 'latitude', 'longitude', 'bathrooms', 'bedrooms',
       'living_area', 'lot_area', 'lot_area_unit', 'tax_assessed_value',
       'zestimate', 'rent_zestimate', 'is_fsba', 'is_non_owner_occupied',
       'days_on_zillow'],
      dtype='object')

In [13]:
dff.to_csv("zillow_cleaned_10.csv", index = False)